In [5]:
import os
import torch
import numpy as np
from base64 import b64decode
from PIL import Image

from datasets import load_dataset

from transformers.utils.import_utils import is_flash_attn_2_available
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor

from colpali_engine.models import ColQwen2, ColQwen2Processor
#from colpali_engine.models import ColPali, ColPaliProcessor # uncomment if you prefer to use ColPali models instead of ColQwen2 models

import weaviate
from weaviate.classes.init import Auth
import weaviate.classes.config as wc
from weaviate.classes.config import Configure
from weaviate.classes.query import MetadataQuery

from qwen_vl_utils import process_vision_info
import base64
from io import BytesIO

import matplotlib.pyplot as plt

from colpali_engine.interpretability import (
    get_similarity_maps_from_embeddings,
    plot_all_similarity_maps,
    plot_similarity_map,
)

In [3]:
# Get rid of process forking deadlock warnings.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available(): # If GPU available
    device = "cuda:0"
elif torch.backends.mps.is_available(): # If Apple Silicon available
    device = "mps"
else:
    device = "cpu"

if is_flash_attn_2_available():
    attn_implementation = "flash_attention_2"
else:
    attn_implementation = "eager"

print(f"Using device: {device}")
print(f"Using attention implementation: {attn_implementation}")

Using device: mps
Using attention implementation: eager


In [4]:
model_name = "vidore/colqwen2-v1.0"

# About a 5 GB download and similar memory usage.
model = ColQwen2.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation=attn_implementation,
).eval()

# Load processor
processor = ColQwen2Processor.from_pretrained(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.88s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [6]:
# A convenience class to wrap the embedding functionality 
# of ColVision models like ColPali and ColQwen2 
class ColVision:
    def __init__(self, model, processor):
        """Initialize with a loaded model and processor."""
        self.model = model
        self.processor = processor

    # A batch size of one appears to be most performant when running on an M4.
    # Note: Reducing the image resolution speeds up the vectorizer and produces
    # fewer multi-vectors.
    def multi_vectorize_image(self, img):
        """Return the multi-vector image of the supplied PIL image."""
        image_batch = self.processor.process_images([img]).to(self.model.device)
        with torch.no_grad():
            image_embedding = self.model(**image_batch)
        return image_embedding[0]

    def multi_vectorize_text(self, query):
        """Return the multi-vector embedding of the query text string."""
        query_batch = self.processor.process_queries([query]).to(self.model.device)
        with torch.no_grad():
            query_embedding = self.model(**query_batch)
        return query_embedding[0]

# Instantiate the model to be used below.
colvision_embedder = ColVision(model, processor) # This will be instantiated after loading the model and processor

In [2]:
dataset = load_dataset("weaviate/irpapers-docs")

In [7]:
def convert_base64str_to_PILImage(base64str):
    page_sample = b64decode(base64str)
    page_sample = Image.open(BytesIO(page_sample))
    return page_sample

In [9]:
# Weaviate Cloud Deployment
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY")),
)

print(client.is_ready())

True


In [10]:
collection_name = "IRPapers_ColQwen2"

if client.collections.exists(collection_name):
    collection = client.collections.delete(collection_name)

collection = client.collections.create(
    name=collection_name,
    properties=[
        wc.Property(name="dataset_id", data_type=wc.DataType.TEXT),
    ],
    vector_config=[
        Configure.MultiVectors.self_provided(
            name="colqwen",
            #encoding=Configure.VectorIndex.MultiVector.Encoding.muvera(),
            vector_index_config=Configure.VectorIndex.hnsw(
                multi_vector=Configure.VectorIndex.MultiVector.multi_vector()
            )
    )]
)

print(len(collection))

/opt/homebrew/lib/python3.11/site-packages/weaviate/warnings.py:236: DeprecationWarning: Dep027: You are using the `multi_vector` argument in `Configure.VectorIndex.hnsw()`, which is deprecated.
            Use the `multi_vector` argument inside `Configure.MultiVectors.module()` instead.
            
  warnings.warn(


0


In [11]:
len(dataset["train"])

3230

In [12]:
import time

page_images = {}

start = time.time()
with collection.batch.dynamic() as batch:
    for i in range(len(dataset["train"])):
        page_image = dataset["train"][i]["base64_str"]
        processed_patch_image = convert_base64str_to_PILImage(page_image)
        
        # Save for `display()` lookup
        page_images[dataset["train"][i]["dataset_id"]] = processed_patch_image
        
        batch.add_object(
            properties={
                "dataset_id": dataset["train"][i]["dataset_id"],
                },
            vector={"colqwen": colvision_embedder.multi_vectorize_image(
                processed_patch_image
            ).cpu().float().numpy().tolist()})

        if i % 10 == 9:
            print(f"Added {i+1} Page objects to Weaviate.")
            print(f"Time taken: {time.time() - start} seconds.")

    batch.flush()

Added 10/1 Page objects to Weaviate.
Time taken: 74.11769700050354 seconds.
Added 20/1 Page objects to Weaviate.
Time taken: 136.6005220413208 seconds.
Added 30/1 Page objects to Weaviate.
Time taken: 197.54818511009216 seconds.
Added 40/1 Page objects to Weaviate.
Time taken: 257.5691668987274 seconds.
Added 50/1 Page objects to Weaviate.
Time taken: 318.00668811798096 seconds.
Added 60/1 Page objects to Weaviate.
Time taken: 378.11646485328674 seconds.
Added 70/1 Page objects to Weaviate.
Time taken: 438.35655879974365 seconds.
Added 80/1 Page objects to Weaviate.
Time taken: 500.49381494522095 seconds.
Added 90/1 Page objects to Weaviate.
Time taken: 562.7690398693085 seconds.
Added 100/1 Page objects to Weaviate.
Time taken: 624.9856331348419 seconds.
Added 110/1 Page objects to Weaviate.
Time taken: 687.0824749469757 seconds.
Added 120/1 Page objects to Weaviate.
Time taken: 748.9251999855042 seconds.
Added 130/1 Page objects to Weaviate.
Time taken: 810.8780400753021 seconds.
Add

INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "no healthy upstream"
	debug_error_string = "UNKNOWN:Error received from peer ipv4:35.201.124.182:443 {grpc_message:"no healthy upstream", grpc_status:14}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "no healthy upstream"
	debug_error_string = "UNKNOWN:Error received from peer ipv4:35.201.124.182:443 {grpc_status:14, grpc_message:"no healthy upstream"}"
>. Retrying with exponential backoff in 2 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "no healthy upstream"
	debug_error_string = "UNKNOWN:Error received from peer ipv4:35.201.124.182:443 {grpc_message:"no healthy upstream", grpc_sta

Added 800/1 Page objects to Weaviate.
Time taken: 4893.305644989014 seconds.


INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "no healthy upstream"
	debug_error_string = "UNKNOWN:Error received from peer ipv4:35.201.124.182:443 {grpc_status:14, grpc_message:"no healthy upstream"}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "no healthy upstream"
	debug_error_string = "UNKNOWN:Error received from peer ipv4:35.201.124.182:443 {grpc_status:14, grpc_message:"no healthy upstream"}"
>. Retrying with exponential backoff in 2 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "no healthy upstream"
	debug_error_string = "UNKNOWN:Error received from peer ipv4:35.201.124.182:443 {grpc_status:14, grpc_message:"no healthy ups

Added 810/1 Page objects to Weaviate.
Time taken: 4953.111788988113 seconds.


...

Added 2320/1 Page objects to Weaviate.
Time taken: 15336.427076101303 seconds.


INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "Received http2 header with status: 502"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"Received http2 header with status: 502", grpc_status:14}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "Received http2 header with status: 502"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"Received http2 header with status: 502", grpc_status:14}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "Received http2 header with status: 502"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:14, gr

Added 2330/1 Page objects to Weaviate.
Time taken: 15396.092548847198 seconds.


INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "Received http2 header with status: 502"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:14, grpc_message:"Received http2 header with status: 502"}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "Received http2 header with status: 502"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"Received http2 header with status: 502", grpc_status:14}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "Received http2 header with status: 502"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:14, gr

Added 2340/1 Page objects to Weaviate.
Time taken: 15456.588770151138 seconds.
Added 2350/1 Page objects to Weaviate.
Time taken: 15516.907783031464 seconds.
Added 2360/1 Page objects to Weaviate.
Time taken: 15578.01084280014 seconds.
Added 2370/1 Page objects to Weaviate.
Time taken: 15638.670325994492 seconds.
Added 2380/1 Page objects to Weaviate.
Time taken: 15700.901359081268 seconds.
Added 2390/1 Page objects to Weaviate.
Time taken: 15763.798426866531 seconds.
Added 2400/1 Page objects to Weaviate.
Time taken: 15826.083003997803 seconds.
Added 2410/1 Page objects to Weaviate.
Time taken: 15888.194521903992 seconds.
Added 2420/1 Page objects to Weaviate.
Time taken: 15951.814320802689 seconds.
Added 2430/1 Page objects to Weaviate.
Time taken: 16015.8331990242 seconds.
Added 2440/1 Page objects to Weaviate.
Time taken: 16078.9594540596 seconds.
Added 2450/1 Page objects to Weaviate.
Time taken: 16142.398133039474 seconds.
Added 2460/1 Page objects to Weaviate.
Time taken: 16206.

In [13]:
len(collection)

3225